<a href="https://colab.research.google.com/github/manamendraJN/SE4050-Fraud-Detection/blob/CNN-1D/notebooks/CNN1D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SE4050 Deep Learning Assignment
CNN-1D — Credit Card Fraud Detection

## 1. Setup

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
import json, os, copy

from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve, f1_score, precision_score, recall_score, accuracy_score,
    auc as auc_fn
)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

Using device: cuda


## 2. Mount Google Drive and load preprocessed splits

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
PROJECT_DIR = "/content/drive/MyDrive/SE4050-Fraud-Detection"
DATA_DIR    = f"{PROJECT_DIR}/data/processed"
RESULTS_DIR = f"{PROJECT_DIR}/results/cnn1d"
CONFIGS_DIR = f"{PROJECT_DIR}/configs"

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CONFIGS_DIR, exist_ok=True)

train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
val_df   = pd.read_csv(f"{DATA_DIR}/val.csv")
test_df  = pd.read_csv(f"{DATA_DIR}/test.csv")

print("Train:", train_df.shape, "Val:", val_df.shape, "Test:", test_df.shape)
for name, split in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"  {name} fraud rate: {split['Class'].mean()*100:.3f}%")

Train: (198608, 31) Val: (42559, 31) Test: (42559, 31)
  train fraud rate: 0.167%
  val fraud rate: 0.167%
  test fraud rate: 0.167%


In [4]:
TARGET_COL = "Class"
feature_cols = [c for c in train_df.columns if c != TARGET_COL]
N_FEATURES = len(feature_cols)
print(f"Using {N_FEATURES} features:", feature_cols)

def to_xy(df):
    X = df[feature_cols].values.astype(np.float32)
    y = df[TARGET_COL].values.astype(np.float32)
    return X, y

X_train, y_train = to_xy(train_df)
X_val,   y_val   = to_xy(val_df)
X_test,  y_test  = to_xy(test_df)

print("Shapes:", X_train.shape, X_val.shape, X_test.shape)

BATCH_SIZE = 2048
def make_loader(X, y, shuffle):
    ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, drop_last=False)

train_loader = make_loader(X_train, y_train, shuffle=True)

Using 30 features: ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount_log']
Shapes: (198608, 30) (42559, 30) (42559, 30)


## 3. Prepare features/target for CNN-1D

In [5]:
TARGET_COL = "Class"
feature_cols = [c for c in train_df.columns if c != TARGET_COL]
N_FEATURES = len(feature_cols)
print(f"Using {N_FEATURES} features:", feature_cols)

def to_xy(df):
    X = df[feature_cols].values.astype(np.float32)
    y = df[TARGET_COL].values.astype(np.float32)   # float for BCEWithLogitsLoss
    return X, y

X_train, y_train = to_xy(train_df)
X_val,   y_val   = to_xy(val_df)
X_test,  y_test  = to_xy(test_df)

print("Shapes:", X_train.shape, X_val.shape, X_test.shape)

BATCH_SIZE = 2048
def make_loader(X, y, shuffle):
    ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, drop_last=False)

train_loader = make_loader(X_train, y_train, shuffle=True)

Using 30 features: ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount_log']
Shapes: (198608, 30) (42559, 30) (42559, 30)


## 4. Define and train CNN-1D

**Architecture**

| Block | Layers |
|---|---|
| Input | `(B, N_FEATURES)` → unsqueeze → `(B, 1, N_FEATURES)` |
| Conv block 1 | Conv1d(1→32, k=3) → BatchNorm → ReLU → MaxPool(2) |
| Conv block 2 | Conv1d(32→64, k=3) → BatchNorm → ReLU → MaxPool(2) |
| Conv block 3 | Conv1d(64→128, k=3) → BatchNorm → ReLU |
| Global pooling | concat(GlobalAvgPool, GlobalMaxPool) → 256 |
| Head | Linear(256→64) → ReLU → Dropout(0.3) → Linear(64→1) |

**Imbalance handling:** `BCEWithLogitsLoss(pos_weight=5)`, matching the `{0:1, 1:5}` class weights used in the tuned TabNet run so the comparison is fair.

**Training:** Adam (lr 1e-3), StepLR, batch size 2048, up to 250 epochs, early stopping on validation ROC-AUC (patience 30), best weights restored.

In [6]:
class CNN1D(nn.Module):
    def __init__(self, channels=(32, 64, 128), kernel_size=3, fc_hidden=64, dropout=0.3):
        super().__init__()
        layers, in_ch = [], 1
        for i, out_ch in enumerate(channels):
            layers += [
                nn.Conv1d(in_ch, out_ch, kernel_size, padding=kernel_size // 2),
                nn.BatchNorm1d(out_ch),
                nn.ReLU(),
            ]
            if i < len(channels) - 1:
                layers.append(nn.MaxPool1d(2))
            in_ch = out_ch
        self.conv = nn.Sequential(*layers)
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.max_pool = nn.AdaptiveMaxPool1d(1)
        self.head = nn.Sequential(
            nn.Linear(in_ch * 2, fc_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(fc_hidden, 1),
        )

    def forward(self, x):
        if x.dim() == 2:            # (B, F) -> (B, 1, F)
            x = x.unsqueeze(1)
        h = self.conv(x)
        h = torch.cat([self.avg_pool(h).squeeze(-1), self.max_pool(h).squeeze(-1)], dim=1)
        return self.head(h).squeeze(-1)   # raw logits

cnn_params = dict(channels=(32, 64, 128), kernel_size=3, fc_hidden=64, dropout=0.3)
model = CNN1D(**cnn_params).to(DEVICE)
print(model)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

CNN1D(
  (conv): Sequential(
    (0): Conv1d(1, 32, kernel_size=(3,), stride=(1,), padding=(1,))
    (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv1d(32, 64, kernel_size=(3,), stride=(1,), padding=(1,))
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv1d(64, 128, kernel_size=(3,), stride=(1,), padding=(1,))
    (9): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
  )
  (avg_pool): AdaptiveAvgPool1d(output_size=1)
  (max_pool): AdaptiveMaxPool1d(output_size=1)
  (head): Sequential(
    (0): Linear(in_features=256, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=64, out_feature

In [7]:
train_params = dict(
    lr=1e-3, weight_decay=1e-5,
    step_size=15, sched_gamma=0.9,
    max_epochs=250, patience=30,
    batch_size=BATCH_SIZE, pos_weight=5.0,
)

criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(train_params["pos_weight"], device=DEVICE))
optimizer = torch.optim.Adam(model.parameters(), lr=train_params["lr"], weight_decay=train_params["weight_decay"])
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=train_params["step_size"], gamma=train_params["sched_gamma"])

@torch.no_grad()
def predict_proba(model, X, batch_size=8192):
    model.eval()
    out = []
    for i in range(0, len(X), batch_size):
        xb = torch.from_numpy(X[i:i + batch_size]).to(DEVICE)
        out.append(torch.sigmoid(model(xb)).cpu().numpy())
    return np.concatenate(out)

@torch.no_grad()
def eval_loss(model, X, y, batch_size=8192):
    model.eval()
    total = 0.0
    for i in range(0, len(X), batch_size):
        xb = torch.from_numpy(X[i:i + batch_size]).to(DEVICE)
        yb = torch.from_numpy(y[i:i + batch_size]).to(DEVICE)
        total += criterion(model(xb), yb).item() * len(xb)
    return total / len(X)

history = {"loss": [], "val_loss": [], "train_auc": [], "val_auc": []}
best_auc, best_state, best_epoch, wait = -np.inf, None, 0, 0

for epoch in range(1, train_params["max_epochs"] + 1):
    model.train()
    running = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        running += loss.item() * len(xb)
    scheduler.step()

    tr_loss = running / len(X_train)
    va_loss = eval_loss(model, X_val, y_val)
    tr_auc  = roc_auc_score(y_train, predict_proba(model, X_train))
    va_auc  = roc_auc_score(y_val,   predict_proba(model, X_val))

    history["loss"].append(tr_loss); history["val_loss"].append(va_loss)
    history["train_auc"].append(tr_auc); history["val_auc"].append(va_auc)

    if va_auc > best_auc:
        best_auc, best_epoch, wait = va_auc, epoch, 0
        best_state = copy.deepcopy(model.state_dict())
    else:
        wait += 1

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d} | loss {tr_loss:.4f} | val_loss {va_loss:.4f} | "
              f"train_auc {tr_auc:.4f} | val_auc {va_auc:.4f}")

    if wait >= train_params["patience"]:
        print(f"Early stopping at epoch {epoch}. Best epoch: {best_epoch} (val_auc={best_auc:.4f})")
        break

model.load_state_dict(best_state)
print(f"Restored best weights from epoch {best_epoch} (val_auc={best_auc:.4f})")

Epoch   1 | loss 0.0381 | val_loss 0.0123 | train_auc 0.9773 | val_auc 0.9626
Epoch   5 | loss 0.0081 | val_loss 0.0123 | train_auc 0.9954 | val_auc 0.9752
Epoch  10 | loss 0.0057 | val_loss 0.0122 | train_auc 0.9991 | val_auc 0.9652
Epoch  15 | loss 0.0053 | val_loss 0.0112 | train_auc 0.9998 | val_auc 0.9787
Epoch  20 | loss 0.0021 | val_loss 0.0203 | train_auc 1.0000 | val_auc 0.9707
Epoch  25 | loss 0.0011 | val_loss 0.0239 | train_auc 1.0000 | val_auc 0.9598
Epoch  30 | loss 0.0019 | val_loss 0.0194 | train_auc 1.0000 | val_auc 0.9706
Epoch  35 | loss 0.0009 | val_loss 0.0281 | train_auc 1.0000 | val_auc 0.9524
Epoch  40 | loss 0.0014 | val_loss 0.0198 | train_auc 1.0000 | val_auc 0.9699
Early stopping at epoch 43. Best epoch: 13 (val_auc=0.9804)
Restored best weights from epoch 13 (val_auc=0.9804)
